# 01 The tolerance

How close does a number have to be to the warehouse's value to count as read back? The first version
of the gate used one absolute tolerance for everything. It passed every claim in the golden set. It
also passed a unit error on every one of them, which is the reason it is gone.

This notebook keeps that dead end and measures the rule that replaced it: compare in the claim's own
unit after the one conversion the catalog allows, within half a unit of the rounding the number was
shown at.

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("READBACK_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))

In [2]:
import yaml
from readback_gate.compare import compare

claims = []
for path in sorted((ROOT / "evals" / "golden").glob("*.yaml")):
    for q in yaml.safe_load(path.read_text())["questions"]:
        for c in q.get("claims", []):
            claims.append({"id": q["id"], **c})
fractions = [c for c in claims if c["unit"] == "fraction"]
len(claims), len(fractions)

(213, 103)

## The two drafts

For every share in the golden set (a fraction in the warehouse, shown as a percent) two drafts are
made from the known value. The right one converts and shows the share in percent. The wrong one
reads the column as if it were already in percent and prints the raw fraction with a percent sign,
the same kind of mistake the catalog made once with a ratio column it had tagged as dollars.

In [3]:
def drafts(c):
    pct = c["value"] * 100
    right = (round(pct, 1), "percent", f"{pct:.1f}%")
    wrong = (round(c["value"], 2), "percent", f"{c['value']:.2f}%")
    return right, wrong

def absolute(claim, evidence, tol=0.5):
    # The first gate: the raw numbers, units ignored, within half a point either way.
    return abs(claim - evidence) <= tol or abs(claim - evidence * 100) <= tol

def readback(claim, unit, display, evidence):
    return compare(claim, unit, evidence, "fraction", display=display).verdict == "agree"

rows = []
for c in fractions:
    right, wrong = drafts(c)
    rows.append({
        "id": c["id"],
        "absolute, right": absolute(right[0], c["value"]),
        "absolute, wrong": absolute(wrong[0], c["value"]),
        "readback, right": readback(*right, c["value"]),
        "readback, wrong": readback(*wrong, c["value"]),
    })
summary = {k: sum(r[k] for r in rows) for k in rows[0] if k != "id"}
summary, len(rows)

({'absolute, right': 103,
  'absolute, wrong': 103,
  'readback, right': 103,
  'readback, wrong': 0},
 103)

The absolute rule passes both drafts of every share: the wrong number sits within half a point of
the raw fraction. It cannot tell a share from a hundredth of it, because it never asks what unit
either number is in. The
readback rule passes the right draft and refuses the wrong one, every time.

In [4]:
example = fractions[0]
right, wrong = drafts(example)
print(example["id"], example.get("label", ""))
print("warehouse value:", example["value"])
for name, d in (("right", right), ("wrong", wrong)):
    r = compare(d[0], d[1], example["value"], "fraction", display=d[2])
    print(f"{name:5} {d[2]:>7}  {r.verdict:9}  {r.reason}")

D-comparison-01 Philadelphia-Camden-Wilmington, PA-NJ-DE-MD
warehouse value: 0.3026470345
right   30.3%  agree      within 0.05 percent (the display rounding of the claim)
wrong   0.30%  disagree   off by 29.9647 percent, more than 0.005


## How tight is tight enough

Half a unit of the display rounding is the tolerance because it is what rounding can explain and no
more. A quarter of a unit refuses honest roundings; a whole unit lets a number that rounds to a
different display through. The same sweep runs on the floor's test answers in the scorecard's
interior tests.

In [5]:
def sweep(scale):
    ok = 0
    for c in fractions:
        right, _ = drafts(c)
        ok += compare(right[0], right[1], c["value"], "fraction", display=right[2], scale=scale).verdict == "agree"
    return ok / len(fractions)

{scale: round(sweep(scale), 3) for scale in (0.25, 0.5, 1.0)}

{0.25: 0.563, 0.5: 1.0, 1.0: 1.0}

At a quarter unit some correctly rounded shares fail: a share that rounds honestly to its display
can sit more than a quarter step away from it. At half a unit every correct draft passes, and the wrong draft above fails at any of
the three scales because the conversion, not the tolerance, is what catches it.

**What changed.** The absolute tolerance is gone. The gate converts first and compares second, and
the tolerance is half the step of the rounding the reader sees. DECISIONS.md has the entry.